# LMIC-CRIS: prostate cancer risk model, final reproducible analysis

Input: `df.xlsx` (the de-identified dataset, sheet `Cleaning`, 346 raw records, patient names replaced by `Patient_ID`).
All steps are deterministic (seed 42). Outputs are CSV files with Excel row numbers only, no names.

Method summary: exact duplicates removed, records with invalid PSA, date or age removed, outcome classified from the primary diagnosis line (advisory NB notes are not used for cancer or benign keywords), records without definitive histology excluded from the primary cohort, XGBoost with native missing-value handling and no imputation.

**Primary validation:** five-fold stratified cross-validation grouped by patient (all records kept, no patient split across folds), repeated 20 times. **Secondary:** one record per patient. Class weights are computed inside each training fold. Intervals for AUC and AUC differences use a patient-clustered bootstrap.

In [1]:
import pandas as pd, numpy as np, re, sys
import sklearn, xgboost
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier

RNG = 42
SRC = "df.xlsx"
print("python", sys.version.split()[0], "| xgboost", xgboost.__version__, "| scikit-learn", sklearn.__version__,
      "| pandas", pd.__version__, "| numpy", np.__version__)

python 3.12.3 | xgboost 3.4.1 | scikit-learn 1.8.0 | pandas 3.0.2 | numpy 2.4.4


## 1. Load and remove exact duplicates
Source row = Excel row number (header = row 1). Duplicates are identical across all columns except the running index `Unnamed: 0`.

In [2]:
raw = pd.read_excel(SRC, sheet_name="Cleaning")
raw["source_row"] = np.arange(len(raw)) + 2
print(f"STEP 1  raw records: {len(raw)}")

cols = [c for c in raw.columns if c not in ("Unnamed: 0", "source_row")]
dup = raw.duplicated(subset=cols, keep="first")
df = raw[~dup].copy()
print(f"STEP 2  exact duplicate records removed: {int(dup.sum())}  -> {len(df)}")

STEP 1  raw records: 346
STEP 2  exact duplicate records removed: 74  -> 272


## 2. Parse values and apply the validity filter
A record is valid if both PSA values and both PSA dates parse, PSA date 1 is on or after 2000-01-01 (earlier dates, such as 1970-01-01, are spreadsheet placeholders), PSA date 2 is after PSA date 1, and age is between 30 and 110. PSA values are capped at 100 ng/mL.

In [3]:
def clean_psa(v):
    if pd.isna(v): return np.nan
    v = str(v).lower().replace("ng/ml", "").replace("ng/mls", "").replace(">", "").strip()
    try: return float(v)
    except Exception: return np.nan

df["AGE_n"] = pd.to_numeric(df["AGE"].astype(str).str.extract(r"(\d+)", expand=False), errors="coerce")
df["psa1"] = df["PSA_Value_1"].apply(clean_psa).clip(upper=100)
df["psa2"] = df["PSA_Value_2"].apply(clean_psa).clip(upper=100)
df["d1"] = pd.to_datetime(df["PSA_Date_1"], errors="coerce")
df["d2"] = pd.to_datetime(df["PSA_Date_2"], errors="coerce")
ok = (df["psa1"].notna() & df["psa2"].notna() & df["d1"].notna() & df["d2"].notna()
      & (df["d1"] >= "2000-01-01") & (df["d2"] > df["d1"]) & df["AGE_n"].between(30, 110))
print(f"STEP 3  invalid PSA/date/age removed: {int((~ok).sum())}  -> {int(ok.sum())}")
df["valid"] = ok
valid = df[ok].copy()

STEP 3  invalid PSA/date/age removed: 69  -> 203


## 3. Outcome classification
Cancer and benign keywords are read from the primary diagnosis line only (text before any `NB` note). Adequacy and uncertainty wording is read from the full text, so an inadequate specimen stays excluded even when the note sits in the `NB` section. Records that cannot be classified definitively are excluded from the primary cohort and listed in `pending_records.csv` for clinical review.

In [4]:
def classify(diag):
    full = str(diag).upper()
    s = re.split(r"\bN\.?B\b", full)[0]
    malignant = ("ADENOCARCINOMA" in s or "CARCINOMA" in s or "MALIGN" in s or "SARCOMA" in s)
    benign = ("HYPERPLASIA" in s or "PROSTATITIS" in s or "BENIGN" in s)
    uncertain = bool(re.search(
        r"READ TEXT|INADEQUATE|INSUFFICIENT|NOT REPRESENTATIVE|COULD NOT BE MADE|CANNOT BE RULED|^\s*\?|INTRA-?EPITHE|SUSPICIOUS", s)) \
        or bool(re.search(r"INADEQUATE|INSUFFICIENT|NOT REPRESENTATIVE|COULD NOT BE MADE", full))
    if uncertain and not (malignant and "ADENOCARCINOMA" in s and "GLEASON" in s):
        return "no_definitive_histology"
    if "?" in s:
        return "no_definitive_histology"
    if malignant: return "malignant"
    if benign: return "benign"
    return "no_definitive_histology"

valid["cls"] = valid["Pathological Diagnosis"].apply(classify)
pend = valid[valid["cls"] == "no_definitive_histology"]
print(f"STEP 4  no definitive histology, excluded from primary cohort: {len(pend)}")
print("        Excel rows:", sorted(pend["source_row"].tolist()))
pend[["source_row", "Pathological Diagnosis"]].to_csv("pending_records.csv", index=False)

STEP 4  no definitive histology, excluded from primary cohort: 21
        Excel rows: [10, 108, 121, 131, 158, 181, 208, 211, 269, 272, 276, 284, 292, 295, 313, 323, 331, 332, 334, 335, 345]


## 4. Features
PSA velocity (ng/mL/yr) and doubling time from the two PSA readings. Prostate volume is extracted from free-text imaging findings and counted only when an explicit mL or cm3 unit is present (cm2 is an area, not a volume). Missing volumes are left missing: XGBoost handles them natively, with no imputation.

In [5]:
df = valid
df["time_yr"] = (df["d2"] - df["d1"]).dt.days / 365.25
df["vel"] = (df["psa2"] - df["psa1"]) / df["time_yr"]
df["dbl"] = np.where(df["vel"] > 0, 0.693 / (df["vel"] / df["psa1"]), np.nan)
ML_RE = re.compile(r"(\d+\.?\d*)\s?(mls|ml|cm3|cm³)\b", re.I)
df["vol"] = df["Imaging_Findings"].apply(
    lambda t: float(ML_RE.search(str(t)).group(1)) if pd.notna(t) and ML_RE.search(str(t)) else np.nan)
df["imaging_susp"] = df["Imaging_Findings"].str.contains(
    "enlarged|lesion|nodule|hypoechoic|remarkable|enlarge", case=False, na=False).astype(int)
df["HTN"] = df["Comorbidities"].str.contains("HTN|HHDx", case=False, na=False).astype(int)
df["T2DM"] = df["Comorbidities"].str.contains("T2DM|DM|diabetes", case=False, na=False).astype(int)
FEATS = ["AGE_n", "psa1", "psa2", "vel", "dbl", "vol", "imaging_susp", "HTN", "T2DM"]

prim = df[df["cls"] != "no_definitive_histology"]
vol_miss = prim["vol"].isna().sum()
print(f"STEP 5  primary cohort: {len(prim)}  "
      f"({(prim['cls']=='malignant').sum()} malignant / {(prim['cls']=='benign').sum()} benign)")
print(f"        prostate volume missing (mL/cm3 only): {vol_miss}/{len(prim)} = {100*vol_miss/len(prim):.1f}%")

STEP 5  primary cohort: 182  (107 malignant / 75 benign)
        prostate volume missing (mL/cm3 only): 16/182 = 8.8%


## 5. Validation functions
Grouped, repeated cross-validation, and a patient-clustered bootstrap for AUC intervals and AUC differences.

In [6]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer

N_REP = 20

def make_model(ytr):
    return XGBClassifier(random_state=RNG, learning_rate=0.1, max_depth=3, n_estimators=50,
                         scale_pos_weight=(ytr == 0).sum() / (ytr == 1).sum(), eval_metric="logloss")

def grouped_cv(X, y, groups, impute=False, n_rep=N_REP):
    """Stratified group k-fold (k=5), repeated. Returns fold AUCs (list) and out-of-fold predictions averaged over repeats."""
    folds, oof = [], np.zeros(len(y))
    for r in range(n_rep):
        cv = StratifiedGroupKFold(5, shuffle=True, random_state=RNG + r)
        for tr, te in cv.split(X, y, groups):
            Xtr, Xte = X[tr], X[te]
            if impute:
                imp = SimpleImputer(strategy="median").fit(Xtr); Xtr, Xte = imp.transform(Xtr), imp.transform(Xte)
            m = make_model(y[tr]).fit(Xtr, y[tr])
            p = m.predict_proba(Xte)[:, 1]
            folds.append(roc_auc_score(y[te], p)); oof[te] += p / n_rep
    return np.array(folds), oof

def cluster_boot(y, groups, scores, ref=None, n=2000):
    """AUC with patient-clustered percentile bootstrap CI for each score; difference vs `ref` if given."""
    pats = np.unique(groups); idx = {q: np.where(groups == q)[0] for q in pats}
    rng = np.random.default_rng(RNG); draws = {k: [] for k in scores}; diffs = {k: [] for k in scores}
    done = 0
    while done < n:
        ch = rng.choice(pats, len(pats)); i = np.concatenate([idx[q] for q in ch])
        if y[i].min() == y[i].max(): continue
        a = {k: roc_auc_score(y[i], np.asarray(s, dtype=float)[i]) for k, s in scores.items()}
        for k in scores:
            draws[k].append(a[k])
            if ref is not None: diffs[k].append(a[k] - a[ref])
        done += 1
    out = []
    for k, s in scores.items():
        row = {"score": k, "auc": round(roc_auc_score(y, np.asarray(s, dtype=float)), 3),
               "ci_low": round(np.percentile(draws[k], 2.5), 3), "ci_high": round(np.percentile(draws[k], 97.5), 3)}
        if ref is not None and k != ref:
            row["diff_vs_" + ref] = round(row["auc"] - roc_auc_score(y, np.asarray(scores[ref], dtype=float)), 3)
            row["diff_ci_low"] = round(np.percentile(diffs[k], 2.5), 3); row["diff_ci_high"] = round(np.percentile(diffs[k], 97.5), 3)
        out.append(row)
    return pd.DataFrame(out)

def summarise(label, folds, n_rec, n_pat, ypos, yneg, pooled=None):
    return {"analysis": label, "records": n_rec, "patients": n_pat, "malignant": int(ypos), "benign": int(yneg),
            "cv_auc_mean": round(float(folds.mean()), 3), "cv_auc_sd": round(float(folds.std()), 3),
            "pooled_oof_auc": None if pooled is None else round(float(pooled), 3)}

## 6. Primary and secondary analyses
Primary: all 182 records, folds grouped by patient. Secondary: first record per patient (139 patients). Bounding: the 21 records without definitive histology forced to each class (grouped, 203 records).

In [7]:
rec = df[df["cls"] != "no_definitive_histology"].copy().reset_index(drop=True)
y_rec = (rec["cls"] == "malignant").astype(int).values
X_rec = rec[FEATS].astype(float).values
g_rec = rec["Patient_ID"].values

f_prim, oof_prim = grouped_cv(X_rec, y_rec, g_rec)
rows = [summarise("PRIMARY: all records, grouped by patient", f_prim, len(rec), rec.Patient_ID.nunique(),
                  y_rec.sum(), (y_rec == 0).sum(), roc_auc_score(y_rec, oof_prim))]

pt = rec[~rec.duplicated(subset=["Patient_ID"], keep="first")].copy().reset_index(drop=True)
y_pt = (pt["cls"] == "malignant").astype(int).values
f_pt, oof_pt = grouped_cv(pt[FEATS].astype(float).values, y_pt, pt["Patient_ID"].values)
rows.append(summarise("SECONDARY: first record per patient", f_pt, len(pt), len(pt), y_pt.sum(), (y_pt == 0).sum(), roc_auc_score(y_pt, oof_pt)))

for label, assign in (("BOUND: uncertain records as malignant", 1), ("BOUND: uncertain records as benign", 0)):
    b = df.copy().reset_index(drop=True)
    yb = np.where(b["cls"] == "no_definitive_histology", assign, (b["cls"] == "malignant").astype(int)).astype(int)
    fb, ob = grouped_cv(b[FEATS].astype(float).values, yb, b["Patient_ID"].values)
    rows.append(summarise(label, fb, len(b), b.Patient_ID.nunique(), yb.sum(), (yb == 0).sum(), roc_auc_score(yb, ob)))
res = pd.DataFrame(rows); res.to_csv("results_summary.csv", index=False)
print(res.to_string(index=False))

                                analysis  records  patients  malignant  benign  cv_auc_mean  cv_auc_sd  pooled_oof_auc
PRIMARY: all records, grouped by patient      182       139        107      75        0.799      0.074           0.802
     SECONDARY: first record per patient      139       139         77      62        0.767      0.082           0.773
   BOUND: uncertain records as malignant      203       157        128      75        0.752      0.080           0.760
      BOUND: uncertain records as benign      203       157        107      96        0.772      0.080           0.779


## 7. Baseline comparison and PSA bands (all 182 records, patient-clustered bootstrap)
AUC of single raw PSA readings as risk scores, and of the model's pooled out-of-fold predictions, with the paired difference from the first PSA. Strata at PSA 1 below or at/above 20 ng/mL are exploratory (cut-point chosen after seeing the data).

In [8]:
scores = {"Model (pooled out-of-fold)": oof_prim, "PSA 1 alone": rec["psa1"].values,
          "PSA 2 alone": rec["psa2"].values, "PSA velocity alone (negated)": -rec["vel"].values}
base = cluster_boot(y_rec, g_rec, scores, ref="PSA 1 alone")
base.to_csv("baseline_comparison.csv", index=False); print(base.to_string(index=False))
rule = rec["psa1"].values >= 4
print("PSA 1 >= 4 ng/mL: sensitivity", round(rule[y_rec == 1].mean(), 3), "| specificity", round((~rule)[y_rec == 0].mean(), 3))

bands = []
for name, mask in (("All records", np.ones(len(rec), bool)), ("PSA 1 below 20 ng/mL", rec["psa1"].values < 20),
                   ("PSA 1 at or above 20 ng/mL", rec["psa1"].values >= 20)):
    t = cluster_boot(y_rec[mask], g_rec[mask], {"Model": oof_prim[mask], "PSA 1": rec["psa1"].values[mask]}, ref="PSA 1")
    m = t.set_index("score")
    bands.append({"stratum": name, "records": int(mask.sum()), "patients": int(len(np.unique(g_rec[mask]))),
                  "malignant": int(y_rec[mask].sum()), "benign": int((y_rec[mask] == 0).sum()),
                  "model_auc": m.loc["Model", "auc"], "model_ci_low": m.loc["Model", "ci_low"], "model_ci_high": m.loc["Model", "ci_high"],
                  "psa1_auc": m.loc["PSA 1", "auc"], "psa1_ci_low": m.loc["PSA 1", "ci_low"], "psa1_ci_high": m.loc["PSA 1", "ci_high"],
                  "diff": m.loc["Model", "diff_vs_PSA 1"], "diff_ci_low": m.loc["Model", "diff_ci_low"], "diff_ci_high": m.loc["Model", "diff_ci_high"]})
bands = pd.DataFrame(bands); bands.to_csv("psa_band_analysis.csv", index=False); print(bands.to_string(index=False))
bc = pd.crosstab(pd.cut(rec["psa1"], [0, 4, 10, 20, 50, 100, 1e6], right=False), rec["cls"]); bc.to_csv("psa1_band_counts_records.csv"); print(bc.to_string())

                       score   auc  ci_low  ci_high  diff_vs_PSA 1 alone  diff_ci_low  diff_ci_high
  Model (pooled out-of-fold) 0.802   0.720    0.877                0.044       -0.035         0.122
                 PSA 1 alone 0.758   0.666    0.838                  NaN          NaN           NaN
                 PSA 2 alone 0.660   0.553    0.758               -0.098       -0.199        -0.009
PSA velocity alone (negated) 0.724   0.630    0.807               -0.034       -0.089         0.021
PSA 1 >= 4 ng/mL: sensitivity 0.907 | specificity 0.133


                   stratum  records  patients  malignant  benign  model_auc  model_ci_low  model_ci_high  psa1_auc  psa1_ci_low  psa1_ci_high  diff  diff_ci_low  diff_ci_high
               All records      182       139        107      75      0.802         0.720          0.877     0.758        0.666         0.838 0.044       -0.035         0.122
      PSA 1 below 20 ng/mL       83        64         31      52      0.598         0.447          0.747     0.431        0.281         0.592 0.167       -0.067         0.400
PSA 1 at or above 20 ng/mL       99        75         76      23      0.856         0.758          0.935     0.823        0.696         0.923 0.033       -0.052         0.133
cls                 benign  malignant
psa1                                 
[0.0, 4.0)              10         10
[4.0, 10.0)             18         11
[10.0, 20.0)            24         10
[20.0, 50.0)            17         11
[50.0, 100.0)            3         21
[100.0, 1000000.0)       3        

## 8. Referral threshold analysis (all 182 records)
Pooled out-of-fold predictions from the primary grouped cross-validation. For each sensitivity target, the highest threshold reaching it is used. Thresholds are chosen on the same predictions that are scored, and class weighting distorts the probabilities, so the figures are optimistic and descriptive only. Sensitivity has a Wilson 95% interval.

In [9]:
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return round(c - h, 3), round(c + h, 3)

trows = []
for target in (0.95, 0.90, 0.80):
    thr = max(t for t in np.unique(oof_prim) if ((oof_prim >= t) & (y_rec == 1)).sum() / (y_rec == 1).sum() >= target)
    pred = oof_prim >= thr
    tp = int((pred & (y_rec == 1)).sum()); fn = int((~pred & (y_rec == 1)).sum()); fp = int((pred & (y_rec == 0)).sum()); tn = int((~pred & (y_rec == 0)).sum())
    lo, hi = wilson(tp, tp + fn)
    trows.append({"target_sensitivity": target, "threshold": round(float(thr), 3), "sensitivity": round(tp / (tp + fn), 3),
                  "sens_ci_low": lo, "sens_ci_high": hi, "specificity": round(tn / (tn + fp), 3),
                  "referred_per_100": round(100 * (tp + fp) / len(y_rec), 1),
                  "benign_not_referred_per_100": round(100 * tn / len(y_rec), 1),
                  "cancers_missed_per_100": round(100 * fn / len(y_rec), 1), "tp": tp, "fn": fn, "fp": fp, "tn": tn})
thr_df = pd.DataFrame(trows); thr_df.to_csv("threshold_analysis.csv", index=False); print(thr_df.to_string(index=False))
print("refer everyone: benign per 100 referred unnecessarily =", round(100 * (y_rec == 0).sum() / len(y_rec), 1))

 target_sensitivity  threshold  sensitivity  sens_ci_low  sens_ci_high  specificity  referred_per_100  benign_not_referred_per_100  cancers_missed_per_100  tp  fn  fp  tn
               0.95      0.147        0.953        0.895         0.980        0.160              90.7                          6.6                     2.7 102   5  63  12
               0.90      0.170        0.907        0.836         0.948        0.187              86.8                          7.7                     5.5  97  10  61  14
               0.80      0.348        0.804        0.719         0.868        0.533              66.5                         22.0                    11.5  86  21  35  40
refer everyone: benign per 100 referred unnecessarily = 41.2


## 9. Missing-data handling comparison (grouped cross-validation, 182 records)
Native handling (primary), median imputation fitted inside each training fold, prostate volume omitted, and complete cases for volume. Paired difference is the mean fold-wise AUC difference from native handling on the same folds. Also lists missing counts per predictor.

In [10]:
miss = pd.DataFrame({"predictor": FEATS,
                     "missing_records": [int(rec[f].isna().sum()) for f in FEATS], "records": len(rec),
                     "missing_patients": [int(pt[f].isna().sum()) for f in FEATS], "patients": len(pt)})
miss["pct_records"] = (100 * miss.missing_records / miss.records).round(1); miss["pct_patients"] = (100 * miss.missing_patients / miss.patients).round(1)
miss.to_csv("predictor_missingness.csv", index=False); print(miss[miss.missing_records > 0].to_string(index=False))

no_vol = [f for f in FEATS if f != "vol"]
strategies = [("Native handling (primary)", FEATS, False, np.ones(len(rec), bool)),
              ("Median imputation inside training folds", FEATS, True, np.ones(len(rec), bool)),
              ("Volume omitted", no_vol, False, np.ones(len(rec), bool)),
              ("Complete cases for volume", FEATS, False, rec["vol"].notna().values)]
out, base_f = [], None
for label, feats, imp, mask in strategies:
    f, o = grouped_cv(rec.loc[mask, feats].astype(float).values, y_rec[mask], g_rec[mask], impute=imp)
    if base_f is None: base_f = f
    out.append({"strategy": label, "records": int(mask.sum()), "malignant": int(y_rec[mask].sum()), "benign": int((y_rec[mask] == 0).sum()),
                "cv_auc_mean": round(float(f.mean()), 3), "cv_auc_sd": round(float(f.std()), 3), "pooled_oof_auc": round(float(roc_auc_score(y_rec[mask], o)), 3),
                "paired_diff_vs_native": None if (mask.sum() != len(rec) or label.startswith("Native")) else round(float((base_f - f).mean()), 3)})
md_df = pd.DataFrame(out); md_df.to_csv("missing_data_comparison.csv", index=False); print(md_df.to_string(index=False))

predictor  missing_records  records  missing_patients  patients  pct_records  pct_patients
      dbl              134      182               100       139         73.6          71.9
      vol               16      182                11       139          8.8           7.9


                               strategy  records  malignant  benign  cv_auc_mean  cv_auc_sd  pooled_oof_auc  paired_diff_vs_native
              Native handling (primary)      182        107      75        0.799      0.074           0.802                    NaN
Median imputation inside training folds      182        107      75        0.805      0.078           0.811                 -0.005
                         Volume omitted      182        107      75        0.776      0.079           0.784                  0.024
              Complete cases for volume      166         98      68        0.816      0.088           0.825                    NaN


## 10. PSA timing sensitivity
The timing of the second PSA relative to biopsy or treatment is not recorded and could not be supplied. (a) PSA dates are compared with the ultrasound date for the first-record patients (ultrasound is not the biopsy, so this is indicative only). (b) The model is refitted without PSA 2, PSA velocity and PSA doubling time (grouped cross-validation, 182 records).

In [11]:
def parse_date(v):
    if pd.isna(v): return pd.NaT
    if hasattr(v, "year"): return pd.Timestamp(v)
    return pd.to_datetime(str(v), dayfirst=True, errors="coerce")

tq = pt.copy(); tq["img"] = tq["Imaging_Date"].map(parse_date); tq = tq[tq["img"].notna()]
trows = []
for name, col in (("PSA 1", "d1"), ("PSA 2", "d2")):
    gap = (tq[col] - tq["img"]).dt.days
    trows.append({"psa": name, "patients_with_imaging_date": len(tq), "psa_before_imaging": int((gap < 0).sum()),
                  "psa_on_or_after_imaging": int((gap >= 0).sum()), "median_days_after_imaging": float(gap.median()),
                  "q1_days": float(gap.quantile(.25)), "q3_days": float(gap.quantile(.75)), "more_than_30_days_after": int((gap > 30).sum())})
timing = pd.DataFrame(trows); timing.to_csv("psa_timing_vs_imaging.csv", index=False); print(timing.to_string(index=False))

nopsa2 = ["AGE_n", "psa1", "vol", "imaging_susp", "HTN", "T2DM"]
f2, o2 = grouped_cv(rec[nopsa2].astype(float).values, y_rec, g_rec)
reduced = pd.DataFrame([{"model": "All nine predictors (primary)", "records": len(rec), "cv_auc_mean": round(float(f_prim.mean()), 3), "cv_auc_sd": round(float(f_prim.std()), 3), "pooled_oof_auc": round(float(roc_auc_score(y_rec, oof_prim)), 3)},
                        {"model": "Without PSA 2, PSA velocity and PSA doubling time", "records": len(rec), "cv_auc_mean": round(float(f2.mean()), 3), "cv_auc_sd": round(float(f2.std()), 3), "pooled_oof_auc": round(float(roc_auc_score(y_rec, o2)), 3)}])
reduced.to_csv("reduced_feature_models.csv", index=False); print(reduced.to_string(index=False))

  psa  patients_with_imaging_date  psa_before_imaging  psa_on_or_after_imaging  median_days_after_imaging  q1_days  q3_days  more_than_30_days_after
PSA 1                         138                  61                       77                        3.0  -223.00   171.50                       62
PSA 2                         138                  45                       93                       81.5   -93.25   310.75                       90


                                            model  records  cv_auc_mean  cv_auc_sd  pooled_oof_auc
                    All nine predictors (primary)      182        0.799      0.074           0.802
Without PSA 2, PSA velocity and PSA doubling time      182        0.783      0.071           0.781


## 11. PSA density (secondary, added after seeing the primary results)
PSA density is PSA 1 divided by prostate volume (defined where volume is observed). Reported as an additional analysis, not part of the primary model: (a) PSA density alone against PSA 1 alone on records with volume; (b) the primary model with PSA density added as a tenth predictor.

In [12]:
rec["psa_dens"] = rec["psa1"] / rec["vol"]
v = rec["vol"].notna().values
d_scores = {"PSA density alone": rec["psa_dens"].values[v], "PSA 1 alone": rec["psa1"].values[v]}
dens = cluster_boot(y_rec[v], g_rec[v], d_scores, ref="PSA 1 alone"); dens["records"] = int(v.sum()); dens["analysis"] = "Single scores, records with volume"
f3, o3 = grouped_cv(rec[FEATS + ["psa_dens"]].astype(float).values, y_rec, g_rec)
extra = pd.DataFrame([{"analysis": "Primary model plus PSA density (all records)", "records": len(rec), "score": "model + density", "auc": round(float(roc_auc_score(y_rec, o3)), 3),
                       "cv_auc_mean": round(float(f3.mean()), 3), "cv_auc_sd": round(float(f3.std()), 3)},
                      {"analysis": "Primary model (all records)", "records": len(rec), "score": "primary", "auc": round(float(roc_auc_score(y_rec, oof_prim)), 3),
                       "cv_auc_mean": round(float(f_prim.mean()), 3), "cv_auc_sd": round(float(f_prim.std()), 3)}])
dens_all = pd.concat([dens, extra], ignore_index=True); dens_all.to_csv("psa_density_analysis.csv", index=False); print(dens_all.to_string(index=False))

            score   auc  ci_low  ci_high  diff_vs_PSA 1 alone  diff_ci_low  diff_ci_high  records                                     analysis  cv_auc_mean  cv_auc_sd
PSA density alone 0.787   0.698    0.867                0.029       -0.015         0.073      166           Single scores, records with volume          NaN        NaN
      PSA 1 alone 0.758   0.664    0.845                  NaN          NaN           NaN      166           Single scores, records with volume          NaN        NaN
  model + density 0.794     NaN      NaN                  NaN          NaN           NaN      182 Primary model plus PSA density (all records)        0.787      0.076
          primary 0.802     NaN      NaN                  NaN          NaN           NaN      182                  Primary model (all records)        0.799      0.074


## 12. Sample for clinician validation of the automated extraction
A random sample of 50 of the 203 valid records (seed 42), including records excluded for no definitive histology, listing the automated classification and extracted volume next to the source text, for a clinician to check. Source row numbers only.

In [13]:
samp = df.sample(n=50, random_state=RNG).sort_values("source_row")
vs = pd.DataFrame({"source_row": samp["source_row"],
                   "diagnosis_text": samp["Pathological Diagnosis"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip().str[:600],
                   "imaging_text": samp["Imaging_Findings"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip(),
                   "automated_class": samp["cls"].map({"malignant": "malignant", "benign": "benign", "no_definitive_histology": "no definitive histology"}),
                   "automated_volume_ml": samp["vol"]})
vs.to_csv("validation_sample_for_clinician.csv", index=False); print(vs["automated_class"].value_counts().to_string())

automated_class
malignant                  25
benign                     18
no definitive histology     7


## 6. Record-level audit (no names)
One row per raw record, showing why each record is in or out of the analysis.

In [14]:
aud = raw[["source_row", "Imaging_Type", "Imaging_Findings"]].copy()
aud["duplicate_removed"] = dup.values
aud = aud.merge(df[["source_row", "valid", "cls", "vol", "Patient_ID"]], on="source_row", how="left")
inv = raw.loc[~dup.values, ["source_row"]].merge(df[["source_row", "valid"]], on="source_row", how="left")
aud["valid"] = aud["valid"].astype("boolean")
def stage(r):
    if r.duplicate_removed: return "removed: exact duplicate"
    if r.valid is False or pd.isna(r.valid): return "removed: invalid PSA/date/age"
    if r.cls == "no_definitive_histology": return "excluded: no definitive histology (clinical review)"
    return "in primary cohort"
aud["status"] = aud.apply(stage, axis=1)
aud["class_used"] = np.where(aud["status"] == "in primary cohort", aud["cls"], "")
aud["volume_ml"] = aud["vol"]
aud["volume_status"] = np.where(aud["status"] == "in primary cohort",
                                np.where(aud["vol"].isna(), "missing", "observed"), "")
first_of_patient = prim.drop_duplicates(subset=["Patient_ID"], keep="first")["source_row"]
aud["in_patient_level_analysis"] = aud["source_row"].isin(first_of_patient)
aud = aud.drop(columns=["valid", "cls", "vol", "Patient_ID", "duplicate_removed"])
aud.to_csv("record_level_audit.csv", index=False)
print(aud["status"].value_counts().to_string())
print("patient-level records:", int(aud["in_patient_level_analysis"].sum()))

status
in primary cohort                                      182
removed: exact duplicate                                74
removed: invalid PSA/date/age                           69
excluded: no definitive histology (clinical review)     21
patient-level records: 139
